# 02 · LangChain con tools — el LLM decide cuándo consultar la BD

En el notebook 01, **siempre** generábamos SQL. Pero algunas preguntas no necesitan la BD ("hola", "cómo te llamas"). En este notebook el LLM recibe una **tool** `consultar_bd` y decide por sí mismo cuándo usarla.

Esto es un paso hacia los **agentes**. El notebook 03 lo lleva al siguiente nivel con LangGraph.

In [1]:
import sys
import os
import tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# BD de prueba (misma receta del notebook 01)
from sqlalchemy import create_engine, text
db_path = Path(tempfile.gettempdir()) / "demo_chat_bd.db"
os.environ["DATABASE_URL"] = f"sqlite:///{db_path}"
engine = create_engine(os.environ["DATABASE_URL"], future=True)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS productos"))
    conn.execute(text("CREATE TABLE productos (id INTEGER PRIMARY KEY, nombre TEXT, precio REAL)"))
    for i, (n, p) in enumerate([("a", 10), ("b", 20), ("c", 30)], start=1):
        conn.execute(text("INSERT INTO productos VALUES (:i, :n, :p)"), {"i": i, "n": n, "p": p})

from chat_bd import config, sql_tools
config.DATABASE_URL = os.environ["DATABASE_URL"]
sql_tools.DATABASE_URL = os.environ["DATABASE_URL"]
sql_tools.crear_engine.cache_clear()
print("BD lista")

BD lista


## 1 · Definir una tool

Una tool es una función Python decorada con `@tool`. LangChain inspecciona su firma y docstring para contarle al LLM qué hace.

In [2]:
from langchain_core.tools import tool
from chat_bd.sql_tools import ejecutar_sql_lectura

@tool
def consultar_bd(sql: str) -> str:
    """Ejecuta una consulta SELECT de SQLite contra la base de datos de pedidos.
    Devuelve el resultado como texto. Solo SELECT, no modifica datos."""
    df = ejecutar_sql_lectura(sql)
    return df.to_string(index=False)

# La tool conserva su descripcion y schema de args. El LLM los vera.
print("nombre:", consultar_bd.name)
print("descripcion:", consultar_bd.description)
print("args:", consultar_bd.args)

nombre: consultar_bd
descripcion: Ejecuta una consulta SELECT de SQLite contra la base de datos de pedidos.
    Devuelve el resultado como texto. Solo SELECT, no modifica datos.
args: {'sql': {'title': 'Sql', 'type': 'string'}}


## 2 · Bind a tool al LLM

`llm.bind_tools([...])` le enseña al LLM qué tools puede llamar. Cuando el LLM responda, podría devolver una **tool call** (una petición para que ejecutemos la tool) en vez de texto.

Para este notebook simulamos la conversación con un LLM falso porque `FakeListChatModel` no implementa tool-calling. **En producción con Ollama/HF funciona de verdad** — el modelo decide.

In [3]:
# Patron "a mano" (lo que un agente haria por ti):
# 1. El LLM mira la pregunta.
# 2. Si necesita datos, devuelve una tool call con el SQL.
# 3. Nosotros ejecutamos la tool y le devolvemos el resultado.
# 4. El LLM formula la respuesta final.

from chat_bd.llm import obtener_llm

pregunta = "cuantos productos hay?"

# Simulacion 1: el LLM decide usar la tool.
llm_decision = obtener_llm(
    proveedor="fake",
    respuestas_fake=["SELECT COUNT(*) AS total FROM productos"],
)
sql_sugerido = llm_decision.invoke(f"Pregunta: {pregunta}").content
print("paso 1 - LLM dice que necesita ejecutar:", sql_sugerido)

paso 1 - LLM dice que necesita ejecutar: SELECT COUNT(*) AS total FROM productos


In [4]:
# Paso 2: ejecutamos la tool.
resultado = consultar_bd.invoke({"sql": sql_sugerido})
print("paso 2 - resultado de la tool:")
print(resultado)

paso 2 - resultado de la tool:
 total
     3


In [5]:
# Paso 3: devolvemos el resultado al LLM para que redacte la respuesta.
llm_respuesta = obtener_llm(
    proveedor="fake",
    respuestas_fake=["Hay 3 productos en la base de datos."],
)
respuesta_final = llm_respuesta.invoke(
    f"Pregunta: {pregunta}\nResultado de la BD: {resultado}\nRedacta la respuesta."
).content
print("paso 3 - respuesta al usuario:", respuesta_final)

paso 3 - respuesta al usuario: Hay 3 productos en la base de datos.


## 3 · Lo que ganas con tools

- **El LLM decide**: si la pregunta no necesita BD, no la llama.
- **Puedes añadir más tools**: `consultar_api_clima`, `enviar_email`, etc.
- **Separación de responsabilidades**: el LLM razona, las tools ejecutan.

## Lo que falta

Este patrón manual funciona para 1 paso. Si el LLM se equivoca (SQL inválido), hay que volver a llamarlo con el error. Cuando tienes **loops** y **estado compartido**, LangChain Expression Language se queda corto. Ahí entra **LangGraph** (notebook 03).